# URL → 9 Ads (any product) — Krea 2

General pipeline: **any product URL** → extract → 9 distinct Identity Edit frames → PIL type overlay → 3×3.

Same Drive stack as face-swap (`headswap_V2`). **No HF login cell.** Models reuse `/content/drive/MyDrive/headswap_V2/models`.

### Hard rules (baked into prompts)
| Rule | Why |
|------|-----|
| Exactly **ONE** person **or** product-only — never both, never clones/triptych | Multi-person frames looked broken |
| Product-only = **floor flat-lay**, no ghost/mannequin/shoes-on-floor person | Ghost + person+props failed |
| Krea draws **visuals only** — no letters | Text via Montserrat overlay |
| Each slot has a **forced unique** pose/crop/camera | Stops 6 identical front stands |
| Type sits **mid-frame**, behind subject when rembg works | Feet placement failed |

### Run
1. Runtime → **GPU**
2. Run **1 → 2 → 3 → 4 → 5** for one URL, **or** run **6** to batch three demo URL types
3. First session mounts Drive; later sessions skip re-download when cache exists


In [ ]:
# @title 1) Setup — GPU · Drive · reuse headswap cache (no re-download if present)
from pathlib import Path
import os, shutil, subprocess, sys

REPO = Path("/content/headswap_V2")
REPO_URL = "https://github.com/malihashar/headswap_V2.git"
BRANCH = "face-swap-no-mask"
DRIVE_MODELS = Path("/content/drive/MyDrive/headswap_V2/models")

# Flip only if you intentionally want a fresh git pull / weight check
FORCE_REPO_UPDATE = False
FORCE_SETUP_SCRIPT = False


def run(cmd, **kw):
    p = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if p.returncode != 0:
        print((p.stdout or "")[-3000:])
        print((p.stderr or "")[-3000:], file=sys.stderr)
        raise SystemExit(f"FAILED: {' '.join(map(str, cmd))}")
    return p


gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise SystemExit("No GPU. Runtime → Change runtime type → GPU, then re-run.")
print("GPU:", gpu.stdout.strip().splitlines()[0])

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# --- repo: clone once, optional update ---
if REPO.exists() and not (REPO / ".git").is_dir():
    shutil.rmtree(REPO)

if not REPO.exists():
    print("Cloning headswap_V2 (first time only)…")
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)])
elif FORCE_REPO_UPDATE:
    print("Updating repo…")
    run(["git", "-C", str(REPO), "fetch", "origin", BRANCH])
    run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"])
else:
    print("Repo already present — skip clone/fetch")

print("Repo:", subprocess.getoutput(f"git -C {REPO} log --oneline -1"))
os.chdir(REPO)

# --- weights: skip setup_colab if Drive models already look populated ---
marker = DRIVE_MODELS / ".url_ad_krea2_ready"
has_drive = DRIVE_MODELS.is_dir() and any(DRIVE_MODELS.iterdir())
if FORCE_SETUP_SCRIPT or not has_drive:
    print("Running setup_colab.sh --krea2 (Drive empty or forced)…")
    run(["bash", "scripts/setup_colab.sh", "--krea2"], cwd=str(REPO))
    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.write_text("ok\n")
else:
    print(f"Drive models present at {DRIVE_MODELS} — skip weight download")
    # still ensure colab env / symlinks without re-fetching if script is fast+idempotent
    env_py = REPO / "scripts" / "colab_env.py"
    if not env_py.exists():
        raise SystemExit(f"Missing {env_py}")
    # light env apply only
    import importlib.util
    spec = importlib.util.spec_from_file_location("colab_env", env_py)
    colab_env = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(colab_env)
    colab_env.apply_env(colab_env.default_paths(use_drive=True))
    colab_env.ensure_import_path(REPO)

check = subprocess.run(
    [sys.executable, "-c", "import numpy; print(numpy.__version__)"],
    capture_output=True, text=True,
)
if check.returncode != 0:
    raise SystemExit("numpy broken — Runtime → Restart session, re-run this cell.")
print("numpy OK:", check.stdout.strip())

# extract + overlay deps
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "requests", "beautifulsoup4", "lxml"],
)
# rembg first (may pull an older/newer Pillow), then force a coherent Pillow
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rembg", "onnxruntime"])
    print("rembg OK")
except Exception as e:
    print("rembg skip (type stays in front):", e)

# Fix Colab PIL mismatch: ImageText imports _Ink missing after rembg/pip churn
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "pillow==11.2.1"],
)
from PIL import Image  # noqa: F401
print("Pillow OK:", __import__("PIL").__version__)

print("Setup complete.")


In [ ]:
# @title 2) Config — one URL or pick a demo type
from pathlib import Path

# Three *different product types* on a Colab-friendly Shopify demo (JSON-LD works)
DEMO_URLS = {
    "apparel_denim": "https://satoshi-demo.myshopify.com/products/classic-straight-jeans",
    "apparel_shorts": "https://satoshi-demo.myshopify.com/products/pierce-gym-short-msh12",
    "bag_object": "https://satoshi-demo.myshopify.com/products/rolltop-backpack",
}

# --- single-run URL (cells 3–5) ---
PRODUCT_URL = DEMO_URLS["apparel_denim"]
# PRODUCT_URL = "https://YOUR-PDP-HERE"

PRIMARY_IMAGE_INDEX = 0
SEED = 46
OUT_ROOT = Path("/content/url_ad_out")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
FALLBACK_URL = DEMO_URLS["apparel_denim"]

print("PRODUCT_URL:", PRODUCT_URL)
for k, v in DEMO_URLS.items():
    print(f"  {k}: {v}")


In [ ]:
# @title 3) Extract product (JSON-LD / OpenGraph) → square product image
import json, io, re, requests
from typing import Any
from bs4 import BeautifulSoup
from PIL import Image
from IPython.display import display

UA = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}


def _as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]


def _price_from_offers(offers):
    for off in _as_list(offers):
        if not isinstance(off, dict):
            continue
        price = off.get("price") or off.get("lowPrice")
        cur = off.get("priceCurrency") or ""
        if price is not None:
            return f"{cur} {price}".strip()
    return None


def extract_product(url: str) -> dict[str, Any]:
    r = requests.get(url, headers=UA, timeout=25)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")
    brief = {
        "source_url": url,
        "product_name": None,
        "description": None,
        "price": None,
        "benefits": [],
        "reviews": [],
        "images": [],
        "category_guess": "object",
        "extraction_gaps": [],
    }
    for tag in soup.find_all("script", attrs={"type": "application/ld+json"}):
        raw = tag.string or tag.get_text() or ""
        try:
            data = json.loads(raw)
        except Exception:
            continue
        for node in (data if isinstance(data, list) else [data]):
            if not isinstance(node, dict):
                continue
            types = [str(t).lower() for t in _as_list(node.get("@type"))]
            if "product" not in types:
                continue
            brief["product_name"] = brief["product_name"] or node.get("name")
            brief["description"] = brief["description"] or node.get("description")
            cat = node.get("category") or ""
            if isinstance(cat, str) and cat:
                brief["category_guess"] = cat
            for im in _as_list(node.get("image")):
                u = im if isinstance(im, str) else (im.get("url") if isinstance(im, dict) else None)
                if isinstance(u, str) and u.startswith("http") and u not in brief["images"]:
                    brief["images"].append(u)
            price = _price_from_offers(node.get("offers"))
            if price:
                brief["price"] = brief["price"] or price
            for rev in _as_list(node.get("review"))[:3]:
                if isinstance(rev, dict) and rev.get("reviewBody"):
                    brief["reviews"].append({
                        "quote": rev["reviewBody"],
                        "attribution": (
                            (rev.get("author") or {}).get("name")
                            if isinstance(rev.get("author"), dict)
                            else rev.get("author")
                        ),
                    })
    og_title = soup.find("meta", property="og:title")
    og_desc = soup.find("meta", property="og:description")
    og_img = soup.find("meta", property="og:image")
    if og_title and not brief["product_name"]:
        brief["product_name"] = og_title.get("content")
    if og_desc and not brief["description"]:
        brief["description"] = og_desc.get("content")
    if og_img:
        u = og_img.get("content")
        if u and u.startswith("http") and u not in brief["images"]:
            brief["images"].insert(0, u)
    for li in soup.select("li")[:40]:
        t = " ".join(li.get_text(" ", strip=True).split())
        if 20 <= len(t) <= 120 and t not in brief["benefits"]:
            brief["benefits"].append(t)
        if len(brief["benefits"]) >= 6:
            break

    blob = " ".join([
        brief.get("product_name") or "",
        brief.get("description") or "",
        str(brief.get("category_guess") or ""),
    ]).lower()
    apparel_kw = ("jean", "shirt", "dress", "pant", "tee", "hoodie", "jacket", "shoe", "sneaker", "boot", "apparel", "clothing")
    brief["is_apparel"] = any(k in blob for k in apparel_kw)

    if not brief["product_name"]:
        brief["extraction_gaps"].append("product_name")
    if not brief["images"]:
        brief["extraction_gaps"].append("images")
    if not brief["price"]:
        brief["extraction_gaps"].append("price")
    return brief


def download_image(url: str) -> Image.Image:
    resp = requests.get(url, headers=UA, timeout=30)
    resp.raise_for_status()
    return Image.open(io.BytesIO(resp.content)).convert("RGB")


def to_square(img: Image.Image, size=1024) -> Image.Image:
    img = img.convert("RGB")
    w, h = img.size
    side = max(w, h)
    canvas = Image.new("RGB", (side, side), (245, 245, 245))
    canvas.paste(img, ((side - w) // 2, (side - h) // 2))
    return canvas.resize((size, size), Image.Resampling.LANCZOS)


def load_brief_and_product(product_url: str, out_dir: Path, primary_index: int = 0):
    brief = None
    last_err = None
    used = product_url
    for candidate in [product_url, FALLBACK_URL]:
        try:
            b = extract_product(candidate)
            if b.get("product_name") and b.get("images"):
                brief, used = b, candidate
                break
            last_err = f"thin extract for {candidate}: {b.get('extraction_gaps')}"
        except Exception as e:
            last_err = f"{candidate}: {e}"
    assert brief and brief.get("images"), f"extract failed: {last_err}"
    idx = max(0, min(primary_index, len(brief["images"]) - 1))
    product_img = download_image(brief["images"][idx])
    product_sq = to_square(product_img)
    out_dir.mkdir(parents=True, exist_ok=True)
    product_img.save(out_dir / "product_primary.jpg", quality=95)
    product_sq.save(out_dir / "product_square.jpg", quality=95)
    (out_dir / "product.json").write_text(json.dumps(brief, indent=2), encoding="utf-8")
    return brief, product_sq, used


OUT_DIR = OUT_ROOT / "single"
brief, product_sq, PRODUCT_URL = load_brief_and_product(PRODUCT_URL, OUT_DIR, PRIMARY_IMAGE_INDEX)
print(json.dumps({
    "product_name": brief["product_name"],
    "price": brief.get("price"),
    "is_apparel": brief.get("is_apparel"),
    "n_images": len(brief["images"]),
    "benefits": brief.get("benefits", [])[:3],
    "source_url": PRODUCT_URL,
    "gaps": brief.get("extraction_gaps"),
}, indent=2))
display(product_sq.resize((320, 320)))



In [ ]:
# @title 4) Build 9 angle instructions + overlays (rules engine)
NAME = brief.get("product_name") or "Product"
PRICE = brief.get("price") or ""
BENEFITS = brief.get("benefits") or []
REVIEWS = brief.get("reviews") or []


def clean_line(s: str, max_len=42) -> str:
    s = " ".join((s or "").split())
    bad = ("pickup", "shipping", "usually ready", "http", "www.", "cookie", "privacy", "add to cart")
    if not s or any(b in s.lower() for b in bad):
        return ""
    return s[:max_len].rstrip(" ,.-")


def build_angles_and_overlays(brief: dict):
    """Category-aware prompts + overlays. Safe for apparel or objects."""
    name = brief.get("product_name") or "Product"
    price = brief.get("price") or ""
    benefits = brief.get("benefits") or []
    reviews = brief.get("reviews") or []
    is_apparel = bool(brief.get("is_apparel"))

    benefit = clean_line(benefits[0] if benefits else "")
    benefit2 = clean_line(benefits[1] if len(benefits) > 1 else "")
    review = clean_line(reviews[0]["quote"] if reviews else "", 48)

    lock = (
        "Edit this exact product photo into one professional square advertisement photograph. "
        "Keep the product identical to the source (colors, materials, logos, shape). "
        "NO text, NO letters, NO words, NO captions, NO watermarks, NO UI, NO price tags drawn in. "
        "Natural proportions — do not stretch. "
    )
    one_person = (
        " HARD: EXACTLY ONE real person in the frame. "
        "FORBIDDEN: two people, three people, clones, triptych, collage, mirrored doubles, group shot, "
        "extra floating heads, person plus a separate flat-lay in the same image. "
    )
    no_person = (
        " HARD: PRODUCT ONLY — zero people. "
        "FORBIDDEN: any person, face, hands, legs, feet, shoes on a body, mannequin, ghost body, "
        "invisible wearer, headless outfit standing upright, person standing beside the product. "
    )
    flat = (
        no_person
        + " TRUE floor / table flat-lay of the main product from the source, "
        "lying flat, connected, catalog quality, pale neutral surface. No extra props or bags. "
    )
    if is_apparel:
        worn = "wearing the exact outfit / garment from the source photo"
        flat_extra = "Arrange the main garment piece(s) as one connected flat-lay."
    else:
        worn = "with the exact product from the source photo (holding or using it naturally)"
        flat_extra = "Single hero product on the surface — one item only, no duplicates."

    angles = [
        ("01_hero_front", lock + one_person +
         f"Single full-body person {worn}, front view, one hand relaxed, bright studio. One figure only. Leave mid-left quiet."),
        ("02_waist_up", lock + one_person +
         f"EXACTLY one person, WAIST-UP crop only (head to hips) — NOT full body. Slight 3/4 turn, {worn}, soft grey light."),
        ("03_over_shoulder", lock + one_person +
         f"EXACTLY one person, BACK three-quarter full body, looking over ONE shoulder at camera, {worn}, cream studio."),
        ("04_wall_lean", lock + one_person +
         f"EXACTLY one person leaning on a plain wall (shoulder/hip), casual, {worn}. Empty wall mid-left. One figure."),
        ("05_lifestyle", lock + one_person +
         f"EXACTLY one person in a real indoor room (window or brick), candid weight-on-one-leg, {worn}. Not seamless studio."),
        ("06_walk", lock + one_person +
         f"EXACTLY one person mid-stride walking toward camera, low angle, {worn}. Motion — not a static plant. One figure only."),
        ("07_flat_diagonal", lock + flat + flat_extra + " Editorial diagonal composition, soft shadow. Leave mid-left empty."),
        ("08_flat_pair", lock + flat + flat_extra + " Calm horizontal / side-by-side arrangement. Leave mid area empty."),
        ("09_flat_hero", lock + flat + flat_extra + " Bold catalog diagonal hero, pale grey. Leave mid-left empty for price type."),
    ]
    overlays = {
        "01_hero_front": {"headline": name, "sub": benefit, "layout": "mid", "behind": True, "size": "lg"},
        "02_waist_up": {"headline": benefit or name, "sub": name if benefit else "", "layout": "mid", "behind": True, "size": "md"},
        "03_over_shoulder": {"headline": f'“{review}”' if review else name, "sub": name if review else "", "layout": "mid", "behind": True, "size": "md"},
        "04_wall_lean": {"headline": name, "sub": review or benefit, "layout": "mid", "behind": True, "size": "lg"},
        "05_lifestyle": {"headline": name, "sub": benefit, "layout": "mid", "behind": True, "size": "md"},
        "06_walk": {"headline": name, "sub": price, "layout": "mid", "behind": True, "size": "lg"},
        "07_flat_diagonal": {"headline": name, "sub": benefit2 or benefit, "layout": "mid", "behind": False, "size": "md"},
        "08_flat_pair": {"headline": name, "sub": benefit, "layout": "mid", "behind": False, "size": "sm"},
        "09_flat_hero": {"headline": price or name, "sub": name if price else benefit, "layout": "mid", "behind": False, "size": "lg"},
    }
    assert len(angles) == 9
    return angles, overlays


ANGLES, OVERLAYS = build_angles_and_overlays(brief)
print("is_apparel:", brief.get("is_apparel"), "|", NAME, "|", PRICE)
for k, inst in ANGLES:
    print(f"{k:18s} behind={OVERLAYS[k]['behind']} size={OVERLAYS[k]['size']} | {inst[:88]}…")


In [ ]:
# @title 5) Generate 9 + Montserrat mid overlay + 3×3
from pathlib import Path
import importlib.util, inspect, time, urllib.request, subprocess, sys

# Heal broken Pillow (rembg/pip can leave ImageText/_Ink ImportError)
try:
    from PIL import Image, ImageDraw, ImageFont
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "pillow==11.2.1"],
    )
    from PIL import Image, ImageDraw, ImageFont

from IPython.display import display, Markdown

REPO = Path("/content/headswap_V2")
spec = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)
PATHS = colab_env.apply_env(colab_env.default_paths(use_drive=True))
colab_env.ensure_import_path(REPO)

import headswap.comfy.runtime as _rt

def _ensure_prompt_server_fixed(loop):
    import server
    instance = getattr(server.PromptServer, "instance", None)
    if instance is not None:
        return instance
    if "asset_manager" in inspect.signature(server.PromptServer.__init__).parameters:
        from app.assets.manager import default_asset_manager
        return server.PromptServer(loop, default_asset_manager())
    return server.PromptServer(loop)

_rt._ensure_prompt_server = _ensure_prompt_server_fixed

FONT_PATH = Path("/content/Montserrat-Black.ttf")
if not FONT_PATH.exists():
    urllib.request.urlretrieve(
        "https://github.com/JulietaUla/Montserrat/raw/master/fonts/ttf/Montserrat-Black.ttf",
        FONT_PATH,
    )

HAS_REMBG = False
rembg_remove = None
try:
    from rembg import remove as rembg_remove
    HAS_REMBG = True
except Exception:
    pass

from headswap.config import load_config
from headswap.pipelines.krea2 import Krea2IdentityEditPipeline

SIZES = {"lg": (68, 28), "md": (52, 24), "sm": (40, 20)}


def load_font(size: int):
    try:
        return ImageFont.truetype(str(FONT_PATH), size=size)
    except Exception:
        for p in (
            "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        ):
            try:
                return ImageFont.truetype(p, size=size)
            except Exception:
                pass
    return ImageFont.load_default()


def wrap_text(draw, text, font, max_width):
    words = (text or "").split()
    if not words:
        return []
    lines, cur = [], words[0]
    for w in words[1:]:
        trial = f"{cur} {w}"
        if draw.textlength(trial, font=font) <= max_width:
            cur = trial
        else:
            lines.append(cur)
            cur = w
    lines.append(cur)
    return lines


def layout_box(layout: str):
    # mid-photo — never near feet
    if layout == "right":
        return 520, 360, 460
    return 48, 360, 600


def overlay_ad(visual, headline, sub, layout="mid", behind=False, size="md"):
    base = visual.convert("RGBA").resize((1024, 1024), Image.Resampling.LANCZOS)
    text_layer = Image.new("RGBA", base.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(text_layer)
    hs, ss = SIZES.get(size, SIZES["md"])
    font_h, font_s = load_font(hs), load_font(ss)
    x, y, max_w = layout_box(layout)
    headline = (headline or "").strip().upper()
    sub = (sub or "").strip().upper()

    def draw_block(text, font, fill, yy):
        for line in wrap_text(d, text, font, max_w)[:3]:
            d.text((x + 3, yy + 3), line, font=font, fill=(0, 0, 0, 160))
            d.text((x, yy), line, font=font, fill=fill)
            yy = d.textbbox((x, yy), line, font=font)[3] + 6
        return yy

    y = draw_block(headline, font_h, (255, 255, 255, 255), y) if headline else y
    if sub and sub != headline:
        draw_block(sub, font_s, (240, 240, 240, 255), y + 4)

    if behind and HAS_REMBG:
        composed = Image.alpha_composite(base, text_layer)
        cut = rembg_remove(base.convert("RGBA")).resize(base.size, Image.Resampling.LANCZOS)
        return Image.alpha_composite(composed, cut).convert("RGB")
    return Image.alpha_composite(base, text_layer).convert("RGB")


def run_nine(product_sq, angles, overlays, out_dir: Path, seed: int, pipe=None):
    cfg = dict(load_config(str(REPO / "configs" / "krea2_identity_edit.yaml")))
    cfg["seed"] = seed
    cfg["single_edit_steps"] = 4
    cfg["single_edit_cfg"] = 1.0
    cfg["single_edit_denoise"] = 1.0
    cfg["verbose"] = False
    cache = Path("/content/.cache/url_ad_krea2")
    cache.mkdir(parents=True, exist_ok=True)
    if pipe is None:
        print("Loading pipeline…")
        pipe = Krea2IdentityEditPipeline(cfg=cfg, cache_dir=cache)
        print("Pipeline loaded.")
    else:
        pipe.cfg.update(cfg)

    ads = []
    t0 = time.perf_counter()
    for i, (key, instruction) in enumerate(angles):
        print(f"[{i+1}/9] {key}", flush=True)
        angle_dir = out_dir / key
        angle_dir.mkdir(parents=True, exist_ok=True)
        pipe.cfg["seed"] = seed + i
        t = time.perf_counter()
        res = pipe.edit_single_image(product_sq, instruction, out_dir=angle_dir)
        ov = overlays[key]
        final = overlay_ad(
            res["image"], ov.get("headline") or "", ov.get("sub") or "",
            layout=ov.get("layout", "mid"),
            behind=bool(ov.get("behind")),
            size=ov.get("size", "md"),
        )
        final.save(out_dir / f"{key}.png")
        ads.append((key, final))
        print(f"  {time.perf_counter()-t:.0f}s", flush=True)
        display(Markdown(f"### {key}"))
        display(final.resize((280, 280)))

    grid = Image.new("RGB", (1024 * 3, 1024 * 3), (255, 255, 255))
    for i, (_, img) in enumerate(ads):
        r, c = divmod(i, 3)
        grid.paste(img.resize((1024, 1024)), (c * 1024, r * 1024))
    grid_path = out_dir / "grid_3x3.jpg"
    grid.save(grid_path, quality=92)
    display(Markdown("### 3×3"))
    display(grid.resize((768, 768)))
    print("Saved", grid_path, f"| {time.perf_counter()-t0:.0f}s")
    return pipe, grid_path


pipe, grid_path = run_nine(product_sq, ANGLES, OVERLAYS, OUT_DIR, SEED)


In [ ]:
# @title 6) BATCH — 3 product types (reuses pipe + Drive; no re-download)
from IPython.display import display, Markdown

assert "run_nine" in dir() and "build_angles_and_overlays" in dir(), "Run cells 4 and 5 first."

batch_urls = [
    ("apparel_denim", DEMO_URLS["apparel_denim"]),
    ("apparel_shorts", DEMO_URLS["apparel_shorts"]),
    ("bag_object", DEMO_URLS["bag_object"]),
]

seen = set()
unique = []
for label, url in batch_urls:
    if url in seen:
        print(f"SKIP duplicate {label}")
        continue
    seen.add(url)
    unique.append((label, url))

results = []
_pipe = pipe if "pipe" in dir() else None
for bi, (label, url) in enumerate(unique):
    display(Markdown(f"## Batch {bi+1}/{len(unique)} — `{label}`"))
    print(url)
    bdir = OUT_ROOT / f"batch_{label}"
    b_brief, b_sq, used = load_brief_and_product(url, bdir, PRIMARY_IMAGE_INDEX)
    angles, overlays = build_angles_and_overlays(b_brief)
    print("product:", b_brief.get("product_name"), "| apparel:", b_brief.get("is_apparel"))
    _pipe, gpath = run_nine(b_sq, angles, overlays, bdir, SEED + 100 * (bi + 1), pipe=_pipe)
    results.append((label, used, str(gpath), b_brief.get("is_apparel")))

print("\n=== BATCH DONE ===")
for row in results:
    print(row)


## Notes
- **Reuse:** Cell 1 skips clone + `setup_colab.sh` when Drive models already exist. Set `FORCE_SETUP_SCRIPT = True` only if weights are broken.
- **Batch:** Cell 6 needs three *different* working PDPs in `DEMO_URLS`. Replace `footwear_or_alt` / `non_apparel` if extract fails.
- **QA checklist per frame:** one subject · no text baked by Krea · pose/crop matches slot · flats have zero humans · type readable mid-frame.
